In [ ]:
"""
Single VECM Combination Tester
--------------------------------
Test a specific combination of metrics:
- Predicted vs Actual values
- MAPE Full and MAPE First-3 per metric
- Actual vs Predicted plots for each metric
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from statsmodels.tsa.vector_ar.vecm import VECM, select_order, select_coint_rank
from IPython.display import display

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIGURATION  — only edit this section
# ─────────────────────────────────────────────
COMBO_TO_TEST = [
    "Ending Outstanding Loans",
    "Finance Charges",
    "Gross Credit Losses",
]

TRAIN_START = "2019-10-31"
TRAIN_END   = "2025-03-31"
PRED_START  = "2025-04-30"
PRED_END    = "2025-12-31"
MAX_LAGS    = 6
# ─────────────────────────────────────────────


# ── Data prep (assumes df_vecm is already in scope) ──────────────────────────
df_model = df_vecm.copy()
df_model["DATE"] = pd.to_datetime(df_model["DATE"])
df_model = df_model.set_index("DATE").sort_index()

train_df = df_model.loc[TRAIN_START:TRAIN_END, COMBO_TO_TEST]
test_df  = df_model.loc[PRED_START:PRED_END,   COMBO_TO_TEST]
n_pred   = len(test_df)

print(f"Train : {train_df.index[0].date()} → {train_df.index[-1].date()}  ({len(train_df)} months)")
print(f"Test  : {test_df.index[0].date()}  → {test_df.index[-1].date()}   ({n_pred} months)")
print(f"Combo : {COMBO_TO_TEST}\n")


# ── Helpers ───────────────────────────────────────────────────────────────────
def mape(actual, predicted):
    actual, predicted = np.array(actual, dtype=float), np.array(predicted, dtype=float)
    mask = actual != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100


# ── Fit VECM ──────────────────────────────────────────────────────────────────
lag_res    = select_order(train_df, maxlags=MAX_LAGS, deterministic="ci")
lag_order  = lag_res.aic if lag_res.aic and lag_res.aic > 0 else 1

coint_res  = select_coint_rank(train_df, det_order=0,
                               k_ar_diff=lag_order, method="trace", signif=0.05)
coint_rank = coint_res.rank

print(f"Lag order        : {lag_order}")
print(f"Cointegration rank: {coint_rank}")

if coint_rank == 0:
    print("⚠️  No cointegration detected — forcing rank=1. Interpret results with caution.")
    coint_rank = 1

model   = VECM(train_df, k_ar_diff=lag_order, coint_rank=coint_rank, deterministic="ci")
result  = model.fit()

raw      = result.predict(steps=n_pred)
forecast = pd.DataFrame(raw, columns=COMBO_TO_TEST, index=test_df.index)

print("\nVECM fitted successfully.\n")


# ── Results table ─────────────────────────────────────────────────────────────
records = []
for metric in COMBO_TO_TEST:
    actual    = test_df[metric].values
    predicted = forecast[metric].values

    for i, date in enumerate(test_df.index):
        records.append({
            "Metric"   : metric,
            "DATE"     : date.strftime("%Y-%m-%d"),
            "Actual"   : round(actual[i], 4),
            "Predicted": round(predicted[i], 4),
        })

results_df = pd.DataFrame(records)

# MAPE summary
mape_rows = []
for metric in COMBO_TO_TEST:
    actual    = test_df[metric].values
    predicted = forecast[metric].values
    mape_rows.append({
        "Metric"       : metric,
        "MAPE Full %"  : round(mape(actual, predicted), 4),
        "MAPE First3 %": round(mape(actual[:3], predicted[:3]), 4),
    })

mape_df = pd.DataFrame(mape_rows)

print("── MAPE Summary ─────────────────────────────────")
display(mape_df)

print("\n── Predicted vs Actual ──────────────────────────")
display(results_df.pivot(index="DATE", columns="Metric", values=["Actual","Predicted"]))

# Save
results_df.to_csv("vecm_single_combo_predictions.csv", index=False)
mape_df.to_csv("vecm_single_combo_mape.csv", index=False)
print("\nSaved: vecm_single_combo_predictions.csv  |  vecm_single_combo_mape.csv")


# ── Plots ─────────────────────────────────────────────────────────────────────
n_metrics = len(COMBO_TO_TEST)
fig, axes = plt.subplots(n_metrics, 1, figsize=(13, 4 * n_metrics), sharex=False)

if n_metrics == 1:
    axes = [axes]

for ax, metric in zip(axes, COMBO_TO_TEST):
    actual    = test_df[metric].values
    predicted = forecast[metric].values
    dates     = test_df.index

    # Full train series as context (last 24 months)
    context = train_df[metric].iloc[-24:]

    ax.plot(context.index, context.values,
            color="#8899aa", linewidth=1.4, linestyle="--", label="Historical (last 24m)")
    ax.plot(dates, actual,
            color="#2196F3", linewidth=2, marker="o", markersize=5, label="Actual")
    ax.plot(dates, predicted,
            color="#FF5722", linewidth=2, marker="s", markersize=5, linestyle="--", label="Predicted")

    # Shade prediction window
    ax.axvspan(dates[0], dates[-1], alpha=0.06, color="#FF5722")

    # Vertical line separating train/test
    ax.axvline(x=train_df.index[-1], color="gray", linewidth=1, linestyle=":")

    m_full   = mape(actual, predicted)
    m_first3 = mape(actual[:3], predicted[:3])
    ax.set_title(f"{metric}\nMAPE Full: {m_full:.2f}%   |   MAPE First-3: {m_first3:.2f}%",
                 fontsize=11, fontweight="bold", pad=10)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right", fontsize=8)

    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x:,.0f}")
    )
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("Value", fontsize=9)

plt.suptitle(f"VECM — Combination: {', '.join(COMBO_TO_TEST)}",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("vecm_single_combo_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved: vecm_single_combo_plots.png")